# Model evaluation for the RIMS archival pipeline
### Runs in Google Colab, free tier, open-weight models only

---

## Read this first — it decides what you can put in this notebook

**No real accession content goes in here.** Colab is Google's infrastructure.
The corpus this pipeline processes contains personal email, unreviewed
identifiers, medical and tax records, and FERPA-covered student records, and
the entire architecture exists because that material stays on university
hardware. Pasting a real extraction into a Colab cell sends it to Google.

That is not a rule to work around. It is the reason the project looks the way
it does, and breaking it in a test notebook would be worse than not testing.

**So this notebook uses a synthetic corpus** (`workbench/fixtures.py`) built to
reproduce the real corpus's *failure modes* without its content: mangled
Latin-1 encodings, extensions that are not formats, 1993 campus BSMTP mail that
crashed the real parser, extractions that returned only a quoted reply chain,
OCR mush, blank pages, and a ZIP+4 postal code that must not be read as a
social security number. The names and subjects are invented. The shapes are
real, and each one is a defect that actually occurred.

Synthetic results do not establish performance on real archival material. When you have a
candidate, the final confirmation run happens on the university server against
real material — not here.

---

## What you are trying to establish

Not "which model is best" in the abstract. Four specific questions:

1. **Does it hold a schema?** Unattended, across hundreds of calls.
2. **Does it know when it cannot answer?** Given a blank page or a truncated
   extraction, does it decline or invent? This is the one that matters most and
   the one nobody benchmarks.
3. **Is it consistent?** Same input, same answer, at temperature 0.
4. **What does it cost?** In hours for 150,000 files.

Four of these need no labelled data, which is why you can start today. Description quality will use the independent frontier-model judging protocol in
validation/FRONTIER-JUDGING.md. Paid judge execution awaits funding. Technical
metrics can be collected first; they do not establish description quality.


## 1. Environment

Colab GPU availability varies. Inspect the actual GPU and free memory. A 4-bit
7B model is a starting candidate, not a guaranteed fit or free-tier entitlement.

Runtime → Change runtime type → T4 GPU, before running this.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q transformers accelerate bitsandbytes sentencepiece 2>&1 | tail -2
print("ready")


## 2. Get the harness

Everything comes from the repository. Nothing is pasted in.


In [ ]:
!git clone -q https://github.com/terbe2022/RIMS-Archival-Project-.git
import sys, pathlib
REPO = pathlib.Path("RIMS-Archival-Project-")
sys.path.insert(0, str(REPO / "src"))

from workbench import fixtures as F, evaluate as E
corpus = F.build_corpus()
print(F.summary(corpus))


## 3. Look at what you are testing on

Read a few before you run anything. If you cannot tell by eye which of these
should be declined, the model has no chance either — and that is worth knowing
about the test set before you blame the model.


In [ ]:
for r in corpus[:3]:
    print("="*70)
    print(r["filename"], f"({r['lane']})")
    print(r["text"][:260])
    print("EXPECTED:", r["expect"])

print("\n\nFiles that MUST be declined:")
for r in [x for x in corpus if x["expect"].get("abstain")][:4]:
    print(f"  {r['filename']:38} {r['text'][:50]!r}")


## 4. Build the test set

`build_cases` adds degraded copies of the synthetic fixtures on top — blanked, truncated
mid-word, reduced to a quoted header. Those have a known correct answer without
anyone labelling anything, which is how abstention becomes measurable.


In [ ]:
cases = E.build_cases(corpus, n_real=len(corpus))
print(f"{len(cases)} cases, {sum(1 for c in cases if c.expects_abstention)} expect a decline")


## 5. Load a model

Historical candidate list below. Start with Qwen2.5-7B-Instruct only; verify its
model licence and actual memory requirements. Other models need separate
protocol decisions and may have access/licence restrictions.

| Model | Params | Licence | Why include it |
|---|---|---|---|
| `Qwen/Qwen2.5-7B-Instruct` | 7B | Apache 2.0 | Strong structured output |
| `meta-llama/Llama-3.1-8B-Instruct` | 8B | Llama licence — read it | What the pipeline uses now |
| `mistralai/Mistral-7B-Instruct-v0.3` | 7B | Apache 2.0 | Good instruction following |
| `microsoft/Phi-3.5-mini-instruct` | 3.8B | MIT | Much smaller; is it enough? |

The Llama licence has use restrictions. Note them in your write-up — for a
university deployment that is a real consideration, not a formality.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = "Qwen/Qwen2.5-7B-Instruct"

quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                           bnb_4bit_quant_type="nf4")
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=quant,
                                             device_map="auto")
print(MODEL, "loaded")


## 6. The prompt

This is the recovered evaluation prompt; equivalence to the current server
prompt has not been established. Do not improve it before the first run —
you are measuring the models, and changing the prompt at the same time means
you learn nothing about either.

Once you have a baseline, changing one thing at a time is exactly the right
next experiment.


In [ ]:
PROMPT = '''You are cataloguing a file from a university archives.

Describe ONLY what the text below contains. Do not infer what it might be from
the collection it came from. If the text is too short, garbled, or contains no
discernible subject, say so — that is a correct and useful answer, and a
confident summary of unreadable text is worse than none.

Rules:
- title: MAX 10 WORDS, a noun phrase. null if not readable.
- description: 2-3 sentences. What this document IS (a referee report, a draft
  with revisions, list traffic) as well as what it is about.
- subjects: 3-6 short lowercase topic terms.
- genre: correspondence | report | draft | minutes | form | dataset |
  publication | notes | list traffic | administrative | unknown
- readable: false if the text does not support a description.

Reply with ONE JSON object and nothing else. No markdown fences.

--- TEXT BEGINS ---
%s
--- TEXT ENDS ---'''

def make_caller(model, tok, max_new=350):
    def call(case):
        msgs = [{"role": "user", "content": PROMPT % case.text}]
        ids = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                      return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(ids, max_new_tokens=max_new, do_sample=False,
                                 pad_token_id=tok.eos_token_id)
        return tok.decode(out[0][ids.shape[-1]:], skip_special_tokens=True)
    return call

caller = make_caller(model, tok)
print("smoke test:", caller(cases[0])[:200])


## 7. Run it

`repeats=2` with `do_sample=False` is what makes self-consistency measurable.
Two different answers to the same input at greedy decoding is a serious finding
— record it, because a model that contradicts itself cannot be audited.

This cloud notebook builds cases from synthetic fixtures only. Record the actual
case count and runtime; no timing guarantee is established. Start with the
two-case smoke test in validation/FIRST-EXPERIMENT.md before this full-run cell.


In [ ]:
import time
results, attempts_by_model = [], {}

name = MODEL.split("/")[-1]
t0 = time.perf_counter()
attempts = E.run_model(name, caller, cases, repeats=2)
attempts_by_model[name] = attempts
results.append(E.score(attempts, cases))
print(f"{name}: {time.perf_counter()-t0:.0f}s")
print()
print(E.comparison_table(results))


## 8. Reading the table

**JSON% / keys%** — below ~95 needs constrained decoding or a repair pass.
Usually fixable; not a verdict on the model.

**declines% / invents%** — the pair that matters. `invents%` is the share of
unusable inputs the model described anyway. A model at 0% declines and 100%
invents will fill a finding aid with confident fiction about blank pages, and
no aggregate quality score will show it.

**consist%** — below 100 at greedy decoding is a finding, not noise.

**hrs/150k** — the whole personal-papers backlog. Compare against the window
that actually exists. Note also that a slow result may be a serving problem
rather than a model problem: moving from Ollama to vLLM gave 6.22× on identical
hardware in this project's own measurements.


In [ ]:
import json
for s in results:
    print(json.dumps(s, indent=2))

# the failures are more informative than the score
bad = [a for a in attempts if a.parsed and not E.abstained(a.parsed)
       and a.case_id.split("#")[0].startswith(("empty","truncated","boilerplate"))]
print(f"\n{len(bad)} inventions on unusable input. First three:")
for a in bad[:3]:
    print(f"\n  {a.case_id}: {a.parsed.get('title')}")
    print(f"     {str(a.parsed.get('description'))[:160]}")


## 9. Now a person grades the descriptions

The measures above narrow the field. They cannot tell you whether a description
is any good — word overlap against a reference rewards paraphrase and punishes
better phrasing, so this harness does not pretend to.

Grade **blind**, against `docs/description-grading-rubric.md`. Four dimensions,
0-3 each, scored separately and never averaged: accuracy, genre, specificity,
restraint. Any accuracy zero — invented content — disqualifies the model
regardless of everything else.


In [ ]:
sheet = E.grading_sheet(attempts_by_model, cases, limit=15)
key = {}
for row in sheet:
    key.update(row.pop("_key", {}))

import pathlib, json
out = pathlib.Path("evaluation"); out.mkdir(exist_ok=True)
(out/"grading-sheet.json").write_text(json.dumps(sheet, indent=1))
(out/"grading-key.json").write_text(json.dumps(key, indent=1))
print(f"{len(sheet)} files to grade. The key is saved separately — do not open "
      f"it until your grades are recorded.")

for row in sheet[:2]:
    print("="*70); print(row["source"])
    print("INPUT:", row["input_opening"][:220].replace("\n"," "))
    for c in row["candidates"]:
        print(f"\n  [{c['tag']}] {c['title']}\n        {c['description']}")


## 10. Comparing several models

Re-run sections 5 through 7 with a different `MODEL`, appending to `results`.
Restart the runtime between models — Colab does not reliably free GPU memory,
and an out-of-memory error halfway through a run wastes more time than the
restart costs.

When you have three or four rows, section 7's table is the comparison.


In [ ]:
# after each model, before loading the next
import gc, torch
del model; gc.collect(); torch.cuda.empty_cache()
print("freed — now restart the runtime and change MODEL")


## 11. What to hand back

One page. Six things:

1. **The comparison table**, unedited.
2. **Every disqualifying finding** — an invented name, a fabricated date, a
   confident description of a blank page. Quote the actual output. One clear
   instance is enough; you do not need a rate.
3. **A recommendation and a runner-up**, with the condition that would change
   your mind.
4. **Licence notes.** Apache 2.0 and MIT are unrestricted. The Llama licence is
   not. For a university deployment that belongs in the recommendation.
5. **What you could not measure here.** Whether descriptions are *useful to an
   archivist* is a different question from whether they are correct, and it
   needs an archivist. Say so rather than letting the numbers imply it.
6. **Your sample.** How many cases, of what kinds. A result without its sample
   is not reproducible.

### Experiments worth adding if you have time

**Does the intake form actually help?** The pipeline prefixes a collection
context block to every prompt. Nobody has tested whether it improves
description quality. Same files, same model, with and without. This is a
load-bearing assumption in the whole design and it is currently unverified.

**Does a smaller model suffice for triage?** Triage is a coarse decision.
Phi-3.5-mini at 3.8B may be enough for the volume tier even if it is weaker on
description, and that would change the cost model substantially.

**Where does quantisation start to hurt?** 4-bit versus 8-bit on the same
model, same cases. The expectation is that JSON adherence degrades before prose
quality does — worth confirming, because JSON adherence is what matters here.
